In [ ]:
try:
    import dolfinx
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh" -O "/tmp/fenicsx-install.sh" && bash "/tmp/fenicsx-install.sh"
    import dolfinx

In [ ]:
%cd /content
!rm -rf beam_fem_ml

!git clone https://github.com/adnan-math/beam_fem_ml.git
%cd beam_fem_ml

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dolfinx import fem

from fem_dataset import BeamDataset

In [ ]:
beam = BeamDataset()
dataset = BeamDataset(mesh_mode="fixed")

In [ ]:
from dolfinx import mesh, fem
import numpy as np
import matplotlib.pyplot as plt

L = 1.0
mesh_levels = [8, 12, 16, 17, 18, 20, 22, 24, 26]

tip_disp = []

for n in mesh_levels:
    print("Mesh size:", n)

    # isotropic refinement (recommended)
    dataset.nx = 3*n
    dataset.ny = n
    dataset.nz = n

    domain, V, uh = dataset.solve(L, dataset.traction)

    fdim = V.mesh.topology.dim - 1

    def right(x):
        return np.isclose(x[0], L)

    facets = mesh.locate_entities_boundary(V.mesh, fdim, right)
    dofs = fem.locate_dofs_topological(V, fdim, facets)

    # safe extraction
    u = uh.x.array.reshape(-1, 3)
    u_tip = np.mean(u[dofs, 2])

    tip_disp.append(u_tip)

# convergence error
err = np.abs(np.diff(tip_disp))

mesh_mid = mesh_levels[1:]

# displacement plot
plt.figure()
plt.plot(mesh_levels, tip_disp, marker='o')
plt.xlabel("Mesh resolution (n)")
plt.ylabel("Tip displacement")
plt.title("Mesh Convergence (Displacement)")
plt.grid()
plt.show()


## Load Linearity Test



This test verifies whether the finite element model responds proportionally to changes in applied traction, as expected in a linear elasticity formulation. It is used to confirm that the numerical implementation preserves the superposition principle before proceeding to data generation and machine learning.


In [ ]:
dataset = BeamDataset()
data1 = dataset.generate(L=1.0, traction=1000)
data2 = dataset.generate(L=1.0, traction=3000)

u1_max = np.min(data1[:,2])
u2_max = np.min(data2[:,2])

print("Ratio:", u2_max / u1_max)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

L_values = np.linspace(0.5, 1.5, 11)
n_points = 100

all_data = []

# ============================================================
# FEM DATA GENERATION LOOP
# ============================================================
for L in L_values:
    print(f"Running FEM for L = {L:.2f}")

    data_L = dataset.generate(L, n_points=n_points)
    all_data.append(data_L)

# Stack everything
all_data = np.vstack(all_data)

print("Final dataset shape:", all_data.shape)

# ============================================================
# SAVE AS CSV (MOST UNIVERSAL)
# ============================================================
df = pd.DataFrame(all_data, columns=["L", "x", "uz"])



plt.figure()

for L in sorted(df["L"].unique()):
    subset = df[df["L"] == L]
    plt.plot(subset["x"], subset["uz"], label=f"L={L}")

plt.xlabel("x")
plt.ylabel("uz")
plt.title("Beam deflection along centerline")
plt.legend()
plt.grid(True)
plt.show()